# Artificial Neural Network

In [1]:
import sys
print("Python used:", sys.executable)


Python used: /usr/bin/python3


In [2]:
%pip install pandas

In [3]:
%pip install scikit-learn

In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf


In [5]:
from sklearn.datasets import load_breast_cancer
data_bunch = load_breast_cancer(as_frame=True)
dataset = pd.concat([data_bunch.data, data_bunch.target], axis=1)
dataset

     mean radius  mean texture  mean perimeter  mean area  ...  target
0          17.99         10.38          122.80     1001.0  ...       0
1          20.57         17.77          132.90     1326.0  ...       0
..           ...           ...             ...        ...  ...     ...
[569 rows x 31 columns]

In [6]:
X = dataset.drop(columns=['target'])
y = dataset['target']

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

In [8]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [9]:
ann = tf.keras.models.Sequential()

In [10]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu', input_shape=(X_train.shape[1],)))

In [11]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))

In [12]:
ann.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

In [13]:
ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [14]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

ann.fit(
    X_train, y_train,
    validation_split=0.2,
    batch_size=32,
    epochs=100,
    callbacks=[early_stop]
)

Epoch 1/100
12/12 - val_accuracy: 0.9890 - val_loss: 0.0182
Epoch 5/100
12/12 - val_accuracy: 0.9890 - val_loss: 0.0205
Epoch 11/100
12/12 - val_accuracy: 0.9890 - val_loss: 0.0229
(early stopping triggered, restored best weights)


In [15]:
y_pred = ann.predict(X_test)
y_pred = (y_pred > 0.5)
print(np.concatenate((y_pred.reshape(len(y_pred), 1), y_test.values.reshape(len(y_test), 1)), 1))

4/4 - 0s 17ms/step
[[0 0]
 [0 0]
 [1 1]
 ... (full list omitted)]


In [16]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[39  3]
 [ 2 70]]


0.956140350877193

In [17]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
precision = precision_score(y_test, y_pred, pos_label=0)
recall = recall_score(y_test, y_pred, pos_label=0)
f1 = f1_score(y_test, y_pred, pos_label=0)

print("Precision (Malignant):", precision)
print("Recall (Malignant):", recall)
print("F1 Score (Malignant):", f1)

print("Classification Report:")
print(classification_report(y_test, y_pred))

Precision (Malignant): 0.9512195121951219
Recall (Malignant): 0.9285714285714286
F1 Score (Malignant): 0.9397590361445783
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94        42
           1       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114
